In [1]:
import pandas as pd
import json
import os

with open("../config.json") as json_file:
    _LOCAL_CONFIG = json.load(json_file)

einstein_csv_path = _LOCAL_CONFIG["einstein_csv_path"]
einstein_folder_path = _LOCAL_CONFIG["einstein_folder_path"]
dataset_output_path = _LOCAL_CONFIG["output_folder_path"]

df = pd.read_csv(einstein_csv_path)
df.head(2)

,clinical_image_filename,dermoscopic_image_filename,patient_id,lesion_id,anatomical_location,sex,final_diagnosis,age_approx,final_label
0,9271c118-73e2-4851-b1ea-d4698df4947d.jpeg,8b8f977e-2659-4bb8-8ad2-a5f35f7eb810.jpeg,625d9afdbca2781e994556c4,62cc6a6e96465749b9b4bcf1,Lesão Perna esquerda anterior,Masculino,squamous cell carcinoma,47.0,CEC
1,8426ef7c-eaf7-4dc9-bcc3-b6c158cd75d3.jpeg,8d8cc25d-290c-43ae-9997-00120713b63e.jpeg,626feb7a5cee4a0ffff5f5ed,626fecd0a8cebb6f665ed7d1,Lesão Cabeça/Pescoço anterior,Feminino,nevus,25.0,NEVO


In [2]:
part1 = df[['clinical_image_filename', 'patient_id',
       'lesion_id', 'anatomical_location', 'sex', 'final_diagnosis', 'age_approx', 'final_label']]
part1['img-src'] = 'CLINICAL'
part2 = df[['dermoscopic_image_filename', 'patient_id',
       'lesion_id', 'anatomical_location', 'sex', 'final_diagnosis', 'age_approx', 'final_label']].rename(columns={'dermoscopic_image_filename': 'clinical_image_filename'})
part2['img-src'] = 'DERMATOSCOPE'

df = pd.concat([part1, part2], ignore_index=True)

df.head()

,clinical_image_filename,patient_id,lesion_id,anatomical_location,sex,final_diagnosis,age_approx,final_label,img-src
0,9271c118-73e2-4851-b1ea-d4698df4947d.jpeg,625d9afdbca2781e994556c4,62cc6a6e96465749b9b4bcf1,Lesão Perna esquerda anterior,Masculino,squamous cell carcinoma,47.0,CEC,CLINICAL
1,8426ef7c-eaf7-4dc9-bcc3-b6c158cd75d3.jpeg,626feb7a5cee4a0ffff5f5ed,626fecd0a8cebb6f665ed7d1,Lesão Cabeça/Pescoço anterior,Feminino,nevus,25.0,NEVO,CLINICAL
2,470f272e-1880-4a84-9e8c-4b50e427ca9e.jpeg,6272f2b29cc4cc1b84b4b371,6272f2b39986789a8728e547,Lesão Perna direita anterior,Feminino,nevus,46.0,NEVO,CLINICAL
3,fe285938-bffe-4b10-80a8-2b9cd72897f5.jpeg,6272f2b29cc4cc1b84b4b371,6272f3269986789a8728e57d,Lesão Membro superior direito anterior,Feminino,nevus,46.0,NEVO,CLINICAL
4,54b3ca20-aee3-4ac2-8fd3-a3075c1f45d4.jpeg,6274003a9986789a8728e69e,6274003a9cc4cc1b84b4b42c,Lesão Cabeça/Pescoço anterior,Feminino,actinic keratosis,78.0,ACT,CLINICAL


In [3]:
df.rename(columns={'patient_id': 'patient-id', 
                   'clinical_image_filename' : 'img-id', 
                   'lesion_id' : 'lesion-id', 
                   'final_diagnosis' : 'clinicalDiagnostic', 
                   'final_label' : 'clinicalMacroCID',
                   'sex' : 'gender', 
                   'age_approx' : 'age', 
                   'anatomical_location' : 'macroBodyRegion'}, inplace=True)
df.head(2)

,img-id,patient-id,lesion-id,macroBodyRegion,gender,clinicalDiagnostic,age,clinicalMacroCID,img-src
0,9271c118-73e2-4851-b1ea-d4698df4947d.jpeg,625d9afdbca2781e994556c4,62cc6a6e96465749b9b4bcf1,Lesão Perna esquerda anterior,Masculino,squamous cell carcinoma,47.0,CEC,CLINICAL
1,8426ef7c-eaf7-4dc9-bcc3-b6c158cd75d3.jpeg,626feb7a5cee4a0ffff5f5ed,626fecd0a8cebb6f665ed7d1,Lesão Cabeça/Pescoço anterior,Feminino,nevus,25.0,NEVO,CLINICAL


In [4]:
diag_map = {
    'MEL' : 'C43',
    'NEVO' : 'D22',
    'CBC' : 'C80',
    'CEC' : 'C44',
    'ACT' : 'L57',
    'SEBO' : 'L82',
}

df['clinicalMacroCID'] = df['clinicalMacroCID'].map(diag_map)
df.head(2)

,img-id,patient-id,lesion-id,macroBodyRegion,gender,clinicalDiagnostic,age,clinicalMacroCID,img-src
0,9271c118-73e2-4851-b1ea-d4698df4947d.jpeg,625d9afdbca2781e994556c4,62cc6a6e96465749b9b4bcf1,Lesão Perna esquerda anterior,Masculino,squamous cell carcinoma,47.0,C44,CLINICAL
1,8426ef7c-eaf7-4dc9-bcc3-b6c158cd75d3.jpeg,626feb7a5cee4a0ffff5f5ed,626fecd0a8cebb6f665ed7d1,Lesão Cabeça/Pescoço anterior,Feminino,nevus,25.0,D22,CLINICAL


In [5]:
df['usePesticide'] = 'I'
df['familySkinCancerHistory'] = 'I'
df['familyCancerHistory'] = 'I'
df['hasItched'] = 'I'
df['hasGrown'] = 'I'
df['hasHurt'] = 'I'
df['hasChanged'] = 'I'
df['hasBled'] = 'I'
df['hasElevation'] = 'I'
df['histoDiagnostic'] = 'EMPTY'
df['histoMacroCID'] = 'EMPTY'

df.head(2)

,img-id,patient-id,lesion-id,macroBodyRegion,gender,clinicalDiagnostic,age,clinicalMacroCID,img-src,usePesticide,familySkinCancerHistory,familyCancerHistory,hasItched,hasGrown,hasHurt,hasChanged,hasBled,hasElevation,histoDiagnostic,histoMacroCID
0,9271c118-73e2-4851-b1ea-d4698df4947d.jpeg,625d9afdbca2781e994556c4,62cc6a6e96465749b9b4bcf1,Lesão Perna esquerda anterior,Masculino,squamous cell carcinoma,47.0,C44,CLINICAL,I,I,I,I,I,I,I,I,I,EMPTY,EMPTY
1,8426ef7c-eaf7-4dc9-bcc3-b6c158cd75d3.jpeg,626feb7a5cee4a0ffff5f5ed,626fecd0a8cebb6f665ed7d1,Lesão Cabeça/Pescoço anterior,Feminino,nevus,25.0,D22,CLINICAL,I,I,I,I,I,I,I,I,I,EMPTY,EMPTY


In [6]:
df['gender'].unique()

gender_map = {
    'Feminino' : 'F',
    'Masculino' : 'M',
}
df['gender'] = df['gender'].map(gender_map)

In [ ]:
body_map = {
    'Lesão Perna esquerda anterior': 'PERNA',
    'Lesão Cabeça/Pescoço anterior':'PESCOCO',
    'Lesão Perna direita anterior': 'PERNA',
    'Lesão Membro superior direito anterior': 'BRACO',
    'Lesão Peitoral': 'PEITORAL',
    'Lesão Dorso inferior': 'DORSO',
    'Lesão Abdominal': 'ABDOME',
    'Lesão Dorso superior': 'DORSO',
    'Lesão Coxa direita anterior': 'COXA',
    'Lesão Membro superior esquerdo anterior': 'BRACO',
    'Lesão Dorso mão direita': 'MAO',
    'Lesão Membro superior esquerdo posterior': 'BRACO',
    'Lesão Dorso pé esquerdo': 'PE',
    'Lesão Cabeça/Pescoço posterior': 'PESCOCO',
    'Lesão Coxa esquerda anterior': 'COXA',
    'Lesão Membro superior direito posterior': 'BRACO',
    'Lesão Perna esquerda posterior': 'PERNA',
    'Lesão Palma mão direita': 'MAO',
    'Lesão Coxa direita posterior': 'COXA',
    'Lesão Dorso pé direito': 'PE',
    'Lesão Planta pé direita': 'PE',
    'Lesão Dorso mão esquerda': 'MAO',
    'Lesão Palma mão esquerda': 'MAO',
    'Lesão Perna direita posterior': 'PERNA',
    'Lesão Coxa esquerda posterior': 'COXA',
    'Lesão Planta pé esquerdo': 'PE',
}

df['macroBodyRegion'] = df['macroBodyRegion'].map(body_map)

In [8]:
df['img-id'] = df['img-id'].apply(lambda x: os.path.join(einstein_folder_path, 'images', x))

In [9]:
df.to_csv(os.path.join(dataset_output_path, 'einstein_cli_derm_metadata.csv'), index=False)